# Phase 4: Machine Learning for Analytics (预测未来) 🔮

> **🎯 目标**: 不要被 "算法" 吓倒。作为分析师，我们只需要会**调用**模型，而不是**写**模型。
> **Status**: `Level: Beginner`, `Focus: Scikit-Learn Workflow`

## 🛠️ Phase 4 此致敬礼: ML 必备军火库 (Toolkit)

请把这几个函数刻在脑子里，它们是你未来 80% 代码的骨架。

### 1. 数据切分 (Data Splitting)
*   **函数**: `train_test_split`
*   **库**: `from sklearn.model_selection import train_test_split`
*   **作用**: 把数据分成 "课本" (Train) 和 "考卷" (Test)。
*   **固定写法**: `X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)`

### 2. 模型本体 (The Model)
*   **函数**: `LogisticRegression` (逻辑回归 - 说是回归，其实是分类)
*   **库**: `from sklearn.linear_model import LogisticRegression`
*   **作用**: 最简单、最可解释的二分类模型 (Yes/No)。
*   **三板斧**: 
    1. `model = LogisticRegression()` (建)
    2. `model.fit(X_train, y_train)` (练)
    3. `model.predict(X_test)` (测)

### 3. 算分 (Evaluation)
*   **函数**: `accuracy_score`, `confusion_matrix`
*   **库**: `from sklearn.metrics import accuracy_score`
*   **作用**: 看看考了多少分。

---

## 🚀 实战: 谁要离职? (Churn Prediction)

**场景**: 我们有用户的 usage data (使用时长、充值金额)。
**目标**: 预测用户下个月是否会流失 (Churn: 1=走, 0=留)。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. 造数据 (Mock Data)
# 假设我们有 1000 个用户
np.random.seed(42)
n_samples = 1000

# 特征 X: 
# - 使用时长 (Hours): 越长越不容易走
# - 投诉次数 (Complaints): 越多越容易走
usage_hours = np.random.normal(50, 15, n_samples)
complaints = np.random.randint(0, 5, n_samples)

# 标签 y (Ground Truth):
# 如果投诉多且使用少，就有可能流失 (Churn=1)
# 这里我们人为制造一点规律，加上一点随机噪音
score = -0.5 * usage_hours + 10 * complaints + 10 # 简单的线性关系
prob = 1 / (1 + np.exp(-score)) # Sigmoid 变成概率
y = (prob > 0.5).astype(int) # 变成 0/1

# 整理成 DataFrame (这一步是为了让你看得舒服，给模型只要 array 就行)
df = pd.DataFrame({
    'Usage': usage_hours,
    'Complaints': complaints,
    'Churn': y
})

print("数据预览:")
print(df.head())

# ---------------------------------------------------------
# 1.5. 探查数据 (EDA & Correlation)
# "在把食材扔进锅里之前，先尝尝咸淡。" —— 资深数据分析师
# ---------------------------------------------------------
# 很多新手直接就开始 split + fit，然后模型效果不好也不知道为啥。
# 必须先看：特征 X 和 目标 y 有没有关系？(相关性)

print("\n--- Correlation Analysis ---")
correlation = df.corr()['Churn'].sort_values()
print(correlation)
print("----------------------------")
# 预期: 
# - Usage 应该是负相关 (用得越久越不走)
# - Complaints 应该是正相关 (投诉越多越容易走)
# 如果这里系数都很小 (比如 0.01)，那神仙模型也救不了你。

# 💡 思考: 
# 这里我们只看 "相关性" (Correlation)。
# Phase 3 的 A/B Testing 才是用来证明 "因果性" (Causality) 的。
# ML 模型通常只负责预测 (Prediction)，利用相关性就足够了。

In [ ]:
# 2. 切分数据 (Train/Test Split)
# X (试卷题): Usage, Complaints
# y (参考答案): Churn

X = df[['Usage', 'Complaints']]
y = df['Churn']

# ✍️ Your Code Here: 调用 train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


In [ ]:
# 3. 训练模型 (Fit)
# 这一步就是 AI "学习" 的过程

# 1. 实例化 (叫一个机器人来)
model = LogisticRegression()

# 2. 训练 (把课本 X_train 和答案 y_train 丢给它)
model.fit(X_train, y_train)


In [ ]:
# 4. 预测与评估 (Predict & Evaluate)
# 让它做考卷 (X_test)，也就是预测未来的用户会不会走

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"模型准确率: {accuracy:.2%}")

# ---------------------------------------------------------
# 5. 混淆矩阵 (Confusion Matrix) - 为什么只看准确率是不够的？
# ---------------------------------------------------------
# 准确率 (Accuracy) 只是一个总分 (e.g., 99分)。
# 混淆矩阵 (Confusion Matrix) 是 "试卷分析"：
# - 你到底是做错了选择题，还是做错了大题？
# - 你是把 "要走的人" 错判成了 "留下来" (放过坏人)？
# - 还是把 "留下来的人" 错判成了 "要走" (冤枉好人)？

cm = confusion_matrix(y_test, y_pred)
print("\n--- 混淆矩阵 (Confusion Matrix) ---")
print(cm)
print("-----------------------------------")
print("[[TN, FP],")
print(" [FN, TP]]")
# TN: 真没走 (预测没走，实际没走) - 答对了
# TP: 真走了 (预测走了，实际走了) - 答对了
# FP: 虚惊一场 (预测走了，实际没走) - 误报 (Type I Error)
# FN: 漏网之鱼 (预测没走，实际走了) - 漏报 (Type II Error) -> 这种最可怕！因为我们没挽留他。

In [ ]:
# 6. 偷看机器人的脑子 (Feature Importance)
# 逻辑回归的系数 (Coefficients) 告诉我们要害在哪里
coef = pd.DataFrame({
    'Feature': X.columns,
    'Weight': model.coef_[0]
})
print("特征权重 (权重越大，影响越大):")
print(coef)

# 结论:
# 如果 Complaints 的权重是正数 (e.g., 2.5)，说明投诉越多越容易走。
# 如果 Usage 的权重是负数 (e.g., -0.1)，说明用得越多越不容易走。